# Algorithm 7 — Lexical Frustration Classifier

**File:** `src/CopilotScope.Collector/Quality/FrustrationAnalyzer.cs`

A **report-only** (not included in the composite score), LLM-free frustration classifier that
combines five transparent signals per user message.

---

## 1  Per-message Score

$$
f_i = \min\!\left(1,\;
  0.45 \cdot \min(h^s_i, 2)
  + 0.15 \cdot \min(h^m_i, 3)
  + 0.30 \cdot \mathbf{1}[J(x_i, x_{i-1}) \geq 0.6]
  + 0.15 \cdot \mathbf{1}[\text{CAPS}(x_i)]
  + 0.10 \cdot \mathbf{1}[\text{punct}(x_i)]
  + 0.20 \cdot \mathbf{1}[\text{corrective}(x_i)]
\right)
$$

where:
- $h^s_i$ = count of **strong** lexicon markers ("doesn't work", "useless", "wtf", …)
- $h^m_i$ = count of **mild** lexicon markers ("wrong", "again", "stop", …)
- $J(a, b)$ = Jaccard word-set similarity between messages $i$ and $i-1$ (rephrasing signal)
- $\text{CAPS}(x)$ = sustained uppercase ($> 60\%$ uppercase letters over $\geq 12$ letter chars)
- $\text{punct}(x)$ = punctuation burst (`!!`, `??`, or `?!` present)
- $\text{corrective}(x)$ = short corrective reply ($|x| < 20$ chars starting with "no", "wrong", etc.)

---

## 2  Jaccard Word-set Similarity

Tokenize by splitting on whitespace/punctuation, keep tokens with $|t| > 2$:

$$
J(a, b) = \frac{|\mathcal{W}_a \cap \mathcal{W}_b|}{|\mathcal{W}_a \cup \mathcal{W}_b|}
         = \frac{|\mathcal{W}_a \cap \mathcal{W}_b|}{|\mathcal{W}_a| + |\mathcal{W}_b| - |\mathcal{W}_a \cap \mathcal{W}_b|}
$$

---

## 3  Session Frustration Index

The mean alone averages away one very angry message among many calm ones.  The index
is pulled **halfway toward the peak**:

$$
\boxed{F = \bar{f} + 0.5 \cdot (\max_i f_i - \bar{f})}
$$

Interpretation: $F \geq 0.5$ = clear frustration, $F \geq 0.25$ = mild friction, $F < 0.25$ = calm.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

STRONG = [
    "doesn't work", "does not work", "not working", "still broken", "still wrong",
    "wrong again", "useless", "terrible", "wtf", "stop doing", "i give up", "you broke",
    "nie działa", "dalej nie działa", "znowu źle", "bez sensu", "do niczego",
]
MILD = [
    "no,", "no.", "that's not", "that is not", "wrong", "incorrect", "not what i",
    "again", "still", "undo", "revert", "why did you", "i said", "as i said",
    "nie o to", "źle", "popraw", "cofnij", "jeszcze raz",
]

def tokenize(text: str) -> set[str]:
    tokens = re.split(r'[ \n\t,\.!\?;:]', text.lower())
    return {t for t in tokens if len(t) > 2}

def jaccard(a: str, b: str) -> float:
    sa, sb = tokenize(a), tokenize(b)
    if not sa or not sb: return 0.0
    inter = len(sa & sb)
    return inter / (len(sa) + len(sb) - inter)

def message_score(text: str, previous: str | None = None) -> tuple[float, list[str]]:
    lower = text.lower()
    score = 0.0
    reasons = []

    strong_hits = sum(1 for m in STRONG if m in lower)
    if strong_hits > 0:
        score += 0.45 * min(strong_hits, 2)
        reasons.append(f'strong marker x{strong_hits}')

    mild_hits = sum(1 for m in MILD if m in lower)
    if mild_hits > 0:
        score += 0.15 * min(mild_hits, 3)
        reasons.append(f'mild marker x{mild_hits}')

    if previous is not None:
        j = jaccard(previous, lower)
        if j >= 0.6:
            score += 0.30
            reasons.append(f'rephrasing (J={j:.2f})')

    letters = [c for c in text if c.isalpha()]
    if len(letters) >= 12 and sum(1 for c in letters if c.isupper()) / len(letters) > 0.6:
        score += 0.15
        reasons.append('sustained CAPS')

    if '!!' in text or '??' in text or '?!' in text:
        score += 0.10
        reasons.append('punctuation burst')

    if (len(text) < 20 and
            any(lower.startswith(p) for p in ('no', 'nie', 'stop', 'wrong', 'źle'))):
        score += 0.20
        reasons.append('short corrective reply')

    return min(1.0, score), reasons

def session_frustration(messages: list[str]) -> dict:
    scores = []
    prev = None
    details = []
    for msg in messages:
        s, r = message_score(msg, prev)
        scores.append(s)
        details.append({'text': msg[:60], 'score': s, 'reasons': r})
        prev = msg.lower()
    if not scores:
        return {}
    mean = np.mean(scores)
    peak = max(scores)
    index = mean + 0.5 * (peak - mean)
    return {'index': index, 'mean': mean, 'peak': peak, 'details': details}

# --- Unit results ---
conversations = [
    ('Calm productive session', [
        'Can you add a unit test for the login function?',
        'Great, now add error handling to the handler.',
        'Please refactor the query builder.',
    ]),
    ('Mild friction', [
        'Add pagination to the API endpoint.',
        'No, that is not what I meant. Add it to the list endpoint.',
        'Wrong, still not correct.',
    ]),
    ('High frustration', [
        'Fix the login bug.',
        'That still does not work!!',
        "YOU'RE STILL BROKEN, FIX IT NOW",
        'That still does not work!!',
        'useless, I give up',
    ]),
]

for name, msgs in conversations:
    r = session_frustration(msgs)
    label = ('clear frustration' if r['index'] >= 0.5
             else 'mild friction' if r['index'] >= 0.25 else 'calm')
    print(f'\n{name}')
    print(f'  Index={r["index"]:.3f}  Mean={r["mean"]:.3f}  Peak={r["peak"]:.3f}  → {label}')
    for d in r['details']:
        flag = '⚑' if d['score'] >= 0.3 else ' '
        print(f'  {flag} [{d["score"]:.2f}] "{d["text"]}"'
              + (f' — {", ".join(d["reasons"])}' if d['reasons'] else ''))

In [ ]:
# Jaccard similarity heatmap for the rephrasing signal
msgs = [
    'please fix the login function',
    'fix the login function please',
    'the login still does not work',
    'can you add a cache layer',
    'add caching to the response',
]
n = len(msgs)
J = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        J[i, j] = jaccard(msgs[i], msgs[j])

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(J, cmap='Blues', vmin=0, vmax=1)
labels = [f'M{i+1}' for i in range(n)]
ax.set_xticks(range(n)); ax.set_xticklabels(labels)
ax.set_yticks(range(n)); ax.set_yticklabels(labels)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{J[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, label='Jaccard similarity')
ax.set_title('Jaccard word-set similarity (rephrasing detection)')
plt.tight_layout()
plt.savefig('frustration_jaccard.png', dpi=150)
plt.show()

print('Messages:')
for i, m in enumerate(msgs):
    print(f'  M{i+1}: "{m}"')